# Shot study · foundation models on Colab (team notebook)

**For teammates: Runtime > Change runtime type > T4 GPU, then Runtime > Run all. Do not edit anything.** The notebook takes one of the four parts that nobody has claimed yet and prints which one. If a "session crashed" notice appears during setup, that is the planned restart after installing packages: click Run all again.

The four parts are two models (TabFM, TabICL v2) times two directions (train on season S1 and predict S2, and the reverse). Both models are pretrained: nothing is trained here, the model reads labeled context rows and predicts query rows in one forward pass. TabFM weights are released for non-commercial use only.

**What this notebook does.** It tunes the inference settings of the model on the training season, predicts the held-out evaluation panel, and saves every prediction to Google Drive. It computes no held-out score: the panel files it reads have no outcome column, and scoring happens later on one machine in `05_model_comparison.ipynb`.

**What it needs.** The shared Drive folder `shotlab` visible in your My Drive (Shared with me > right-click `shotlab` > Organize > Add shortcut > My Drive).

**Restart-safe.** Predictions are written in 2,000-row chunks under a hash of the full run configuration. Running again skips finished chunks and keeps your part, so a disconnected session loses at most one chunk.

**Several people run this same notebook.** If a banner says automatic saving failed, ignore it: it does not affect the run.

## 0. Setup

In [1]:
MY_PART = ""
PARTS = ["tabfm__S1_to_S2", "tabfm__S2_to_S1", "tabicl__S1_to_S2", "tabicl__S2_to_S1"]
DRIVE = "/content/drive/MyDrive/shotlab"

import os
import pathlib
import subprocess
import sys
import time
import uuid
import zipfile

from google.colab import drive

drive.mount("/content/drive")
if not pathlib.Path(DRIVE, "artifacts", "logs").exists():
    raise RuntimeError('Folder "shotlab" is not in your My Drive. In Google Drive: Shared with me > right-click shotlab > Organize > Add shortcut > My Drive. Then Run all again.')
CLAIMS = pathlib.Path(DRIVE, "artifacts", "logs")
MINE = pathlib.Path("/content/drive/MyDrive/shotlab_my_part.txt")
GREEDY = pathlib.Path("/content/drive/MyDrive/shotlab_greedy.txt").exists()


def tickets(part):
    return sorted(path.name for path in CLAIMS.glob(f"claim__{part}__*"))


def held():
    """Parts this runner claimed and still holds: its ticket is the earliest one for the part."""
    lines = [line.split() for line in MINE.read_text().splitlines()] if MINE.exists() else []
    return [part for part, token in lines if tickets(part)[:1] == [f"claim__{part}__{token}.txt"]]


def claim_part(wait=45):
    """Tickets the first part nobody holds, then checks twice that no earlier ticket turned up (two sessions starting together)."""
    token = f"{time.time():012.0f}_{uuid.uuid4().hex[:8]}"
    for part in PARTS:
        if tickets(part):
            continue
        ticket = CLAIMS / f"claim__{part}__{token}.txt"
        ticket.write_text(token)
        for _ in range(2):
            time.sleep(wait)
            if tickets(part)[0] != ticket.name:
                ticket.unlink()
                break
        else:
            with open(MINE, "a") as handle:
                handle.write(f"{part} {token}\n")
            return part
    return None


def next_part():
    """An unfinished part this runner holds; else, on a first run or for a greedy runner, a newly claimed one."""
    mine = held()
    unfinished = [part for part in mine if not pathlib.Path(DRIVE, f"results_{part}.zip").exists()]
    if unfinished:
        return unfinished[0]
    return mine[-1] if mine and not GREEDY else claim_part()


# claims are remembered in the runner's own Drive, so the restart after installing and later resumes keep the same part
PART = MY_PART or next_part()
if PART is None:
    raise RuntimeError("All four parts are taken. Nothing to run here; tell Brandon.")
assert PART in PARTS, PART
MODEL, direction = PART.split("__")
DIRECTIONS = [direction]
print("YOUR PART:", PART)
ROOT = pathlib.Path("/content/shotlab")
bundle_zip = pathlib.Path(DRIVE) / "shotlab_bundle.zip"
if not bundle_zip.exists():
    # fallback: bundle uploaded in pieces (.part00, .part01, ...); joined on local disk so sessions sharing the folder never write the same file
    bundle_zip = pathlib.Path("/content/shotlab_bundle.zip")
    bundle_zip.write_bytes(b"".join(part.read_bytes() for part in sorted(pathlib.Path(DRIVE).glob("shotlab_bundle.zip.part*"))))
with zipfile.ZipFile(bundle_zip) as archive:
    archive.extractall(ROOT)
os.environ.update(SHOTLAB_ROOT=str(ROOT), SHOTLAB_ARTIFACTS=f"{DRIVE}/artifacts", SHOTLAB_BUNDLE=str(ROOT / "artifacts" / "bundle"))
sys.path.insert(0, str(ROOT / "src"))
sorted(p.name for p in (ROOT / "artifacts" / "bundle").iterdir())

ValueError: mount failed

In [ ]:
# Same versions as the local machine. torch is left as Colab ships it (the CUDA build); its version is recorded with every run.
PINS = ["numpy==2.5.5", "pandas==3.1.1", "pyarrow==26.0.1", "scikit-learn==1.9.1", "pyyaml==6.0.3", {"tabfm": "tabfm[pytorch] @ git+https://github.com/google-research/tabfm@fbb665569425fd2f490c6576b3af967876fe11ff", "tabicl": "tabicl==2.2.0"}[MODEL]]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)

import importlib.metadata
import numpy
import pandas

if (numpy.__version__, pandas.__version__) != (importlib.metadata.version("numpy"), importlib.metadata.version("pandas")):
    print("New numpy/pandas installed. The runtime restarts now; run the notebook again from the top.")
    os.kill(os.getpid(), 9)

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch
import yaml

from shotlab import bundle, inference, store
from shotlab.data import ARTIFACTS, CONFIGS, config, load_grid

pd.set_option("display.width", 220, "display.max_columns", 30)
info = bundle.verify(inference.BUNDLE)
for season in ("S1", "S2", "T"):
    assert "outcome" not in pd.read_parquet(inference.BUNDLE / f"panel_{season}.parquet").columns
assert torch.cuda.is_available(), "Switch the runtime to a GPU: Runtime > Change runtime type."
gpu = torch.cuda.get_device_properties(0)
print("bundle built from commit", info["git_commit"][:7], "· checksums verified · query files carry no outcome")
print("GPU:", gpu.name, f"{gpu.total_memory / 2**30:.0f} GB")
store.environment()

## 1. Timing pilot

Training-season rows only: month block 0 supplies 2,000 queries, the other blocks supply the context. The pilot answers two questions before anything expensive runs: how large a context fits in GPU memory, and how long the frozen grid would take.

In [ ]:
PILOT_SIZES = [4000, 16000, 64000, 128000, 10**9]
pilot_path = ARTIFACTS / "pilot" / f"{MODEL}.csv"
if pilot_path.exists():
    pilot = pd.read_csv(pilot_path)
else:
    runs = []
    for variant in {"tabfm": [{"cache_context": True, "keep_cache_on_device": True}, {"cache_context": False, "keep_cache_on_device": True}, {"cache_context": True, "keep_cache_on_device": False}], "tabicl": [{"kv_cache": False}, {"kv_cache": True}]}[MODEL]:
        runs.append(inference.pilot(MODEL, "S1_to_S2", PILOT_SIZES, n_estimators=8, queries=2000, options=variant).assign(variant=json.dumps(variant)))
    pilot = pd.concat(runs, ignore_index=True)
    pilot_path.parent.mkdir(parents=True, exist_ok=True)
    pilot.to_csv(pilot_path, index=False)
pilot.round(3)

## 2. Run settings

The grid was frozen before the pilot. `docs/freeze.md` allows one kind of change here: cutting the grid or capping the context when the pilot shows the frozen grid cannot run. The rule was written down before any held-out score existed, so every session applies it the same way:

1. `MEMORY_OPTIONS` are the memory settings under which the pilot fitted the largest context (they change memory use and speed, not the model). Context sizes that still did not fit are dropped; "all rows" then means the largest size that fitted.
2. The grid is the first tier below whose projected tuning time fits `TUNING_BUDGET_HOURS` per direction. Every tier is a subset of the frozen grid; the last one is a single fixed setting.

The tier used is saved next to the pilot table and travels with the results.

In [ ]:
TUNING_BUDGET_HOURS = 2.0
MEMORY_OPTIONS = {"tabfm": {"keep_cache_on_device": False}, "tabicl": {}}[MODEL]
TIERS = {
    "frozen": {},
    "reduced": {"m": [100, 1000, 3000], "context_rows": [4000, 16000, 64000], "n_estimators": [8]},
    "small": {"m": [300, 3000], "context_rows": [4000, 16000, 64000], "n_estimators": [8], "sampling": ["uniform"]},
    "fixed": {"m": [3000], "context_rows": [16000], "n_estimators": [8], "sampling": ["uniform"]},
}


def usable(pilot, options):
    matches = pilot["variant"].map(lambda text: all(options.get(k) == v for k, v in json.loads(text).items()))
    return pilot[matches]


def projected_hours(pilot, spec, queries=config()["tuning_query_size"], folds=5):
    """Rough cost of the tuning grid: one fit plus chunked prediction per configuration and fold."""
    ok = usable(pilot, spec.get("options", {}))
    ok = ok[ok["status"] == "ok"].sort_values("context")
    seconds = 0.0
    for point in inference.grid_points(spec, spec.get("options", {}).get("max_context")):
        rows = min(point["context_rows"], ok["context"].max())
        fit = np.interp(rows, ok["context"], ok["fit_s"])
        predict = np.interp(rows, ok["context"], ok["predict_s"]) * queries / 2000
        seconds += folds * (fit + predict) * point["n_estimators"] / 8
    return seconds / 3600


def tier_spec(name):
    spec = yaml.safe_load(frozen_path.read_text())
    spec["options"].update(MEMORY_OPTIONS)
    grid = {**spec["param_grid"], **TIERS[name]}
    tried = usable(pilot, spec.get("options", {}))
    if (tried["status"] != "ok").any():
        cap = int(tried.loc[tried["status"] == "ok", "context"].max())
        spec["options"]["max_context"] = cap
        grid["context_rows"] = [size for size in grid["context_rows"] if size == "max" or size <= cap] or [cap]
    spec["param_grid"] = grid
    return spec


frozen_path = CONFIGS / f"{MODEL}.frozen.yaml"
if not frozen_path.exists():
    frozen_path.write_text((CONFIGS / f"{MODEL}.yaml").read_text())
tiers = pd.DataFrame([{"tier": name, "configurations": len(inference.grid_points(tier_spec(name), tier_spec(name)["options"].get("max_context"))),
                       "tuning_hours_per_direction": projected_hours(pilot, tier_spec(name))} for name in TIERS]).set_index("tier")
fits = tiers.index[tiers["tuning_hours_per_direction"] <= TUNING_BUDGET_HOURS]
TIER = fits[0] if len(fits) else "fixed"
spec = tier_spec(TIER)
(CONFIGS / f"{MODEL}.yaml").write_text(yaml.safe_dump(spec, sort_keys=False))
grid_note = ARTIFACTS / "pilot" / f"{PART}__grid.json"
grid_note.write_text(json.dumps({"tier": TIER, "budget_hours": TUNING_BUDGET_HOURS, "tiers": tiers.reset_index().to_dict("records"), "spec": spec}, indent=2))
tiers.round(2)

In [ ]:
print(f"tier used: {TIER}")
print(yaml.safe_dump(load_grid(MODEL), sort_keys=False))

## 3. Primary run

For each direction in `DIRECTIONS` (train S1 → predict S2, and the reverse):

1. **Tune** on the training season with the same five month blocks the trained models use. Context rows come from four blocks, queries from the fifth. Smoothed shooter and defender rates are cross-fitted, so no context row carries its own outcome.
2. **Select** the simplest configuration within one standard error of the best log loss.
3. **Calibrate** with a two-parameter Platt step fitted on the out-of-fold predictions of the selected configuration.
4. **Predict** the 50,000-shot evaluation panel of the other season, once per context seed.

In [ ]:
log_path = ARTIFACTS / "logs" / f"{PART}.log"
log_path.parent.mkdir(parents=True, exist_ok=True)


def progress(line):
    stamped = f"{time.strftime('%m-%d %H:%M:%S')} {line}"
    print(stamped)
    with open(log_path, "a") as handle:
        handle.write(stamped + "\n")


inference.run_suite("primary", MODEL, DIRECTIONS, progress)

### What tuning chose

Scores below are month-blocked scores inside the training season. They say nothing yet about the held-out season.

In [ ]:
rows, tables = [], {}
for direction in DIRECTIONS:
    selection = inference.load_selection(MODEL, direction)
    rows.append({"direction": direction, **selection["best_params"], "cv_log_loss": selection["cv_log_loss_selected"],
                 "cv_log_loss_best": selection["cv_log_loss_best"], "cv_auc": selection["cv_auc_selected"],
                 "platt_slope": selection["platt"]["slope"], "platt_intercept": selection["platt"]["intercept"],
                 "configurations": selection["configurations"], "tuning_hours": selection["tuning_seconds"] / 3600})
    tables[direction] = pd.read_parquet(inference.selection_path(inference.primary_config(MODEL, direction)).with_suffix(".parquet"))
pd.DataFrame(rows).set_index("direction").T

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(tables), figsize=(5.5 * len(tables), 3.6), sharey=True, squeeze=False)
axes = axes.ravel()
for ax, (direction, table) in zip(axes, tables.items()):
    mean = table.groupby(["param_context_rows", "param_n_estimators", "param_sampling", "param_m"])["log_loss"].mean()
    best_m = mean.groupby(level=[0, 1, 2]).min().reset_index()
    sizes = sorted(best_m["param_context_rows"].unique())
    for (members, sampling), part in best_m.groupby(["param_n_estimators", "param_sampling"]):
        ax.plot([sizes.index(size) for size in part["param_context_rows"]], part["log_loss"], marker="o",
                label=f"{members} members, {sampling}")
    ax.set_xticks(range(len(sizes)), ["all" if size >= 10**8 else f"{size // 1000}k" for size in sizes])
    ax.set_title(f"{direction}: inner CV log loss (best m)")
    ax.set_xlabel("context rows")
axes[0].set_ylabel("log loss")
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()

## 4. Explanation probes

Predictions on 2,000 probe shots with feature blocks shuffled and single features shifted. The same probe and the same perturbations are applied to all four models; importance and effect curves are computed locally, where the outcomes are.

In [ ]:
inference.run_suite("explain", MODEL, DIRECTIONS, progress)

## 5. Side arms

Structure settings stay at the primary selection. Only the smoothing strength `m` is re-tuned, and only where the arm contains smoothed rates.

- **Raw IDs:** shooter and defender IDs as plain categorical columns instead of smoothed rates.
- **Ablation ladder:** location → + shot type → + creation → + defense → + closeout, and the primary set plus team and month.
- **Learning curve:** nested training subsets of 500 to 100,000 shots, the whole subset as context.
- **As shipped (TabFM parts only):** the built-in Platt calibration of TabFM switched on, which spends part of the context on calibration. Reported next to the primary arm, which uses the same external Platt step as every other model.

In [ ]:
SIDE_ARMS = {"tabfm": ["as_shipped"], "tabicl": []}[MODEL] + ["raw_ids", "ablation", "learning_curve"]
for suite in SIDE_ARMS:
    inference.run_suite(suite, MODEL, DIRECTIONS, progress)

## 6. Test season (optional)

Context: both labeled seasons. Queries: the whole unlabeled test season. It needs the primary selections of both directions in the Drive folder. Run only if the pilot shows it is affordable; otherwise the trained models supply the test predictions.

In [ ]:
RUN_TEST_SEASON = False
if RUN_TEST_SEASON:
    inference.run_suite("final", MODEL, progress=progress)

## 7. Pack the results

Parquet, JSON and CSV only. No model objects and no pickles cross machines.

In [ ]:
import hashlib

catalog = store.catalog()
catalog = catalog[catalog["model"] == MODEL]
display(catalog.groupby(["arm", "direction", "role"])["rows"].agg(["size", "sum"]).rename(columns={"size": "files", "sum": "rows"}))
out = pathlib.Path(DRIVE) / f"results_{PART}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in ("predictions", "tuning", "explain"):
        for path in sorted((ARTIFACTS / folder / MODEL).glob("*")):
            archive.write(path, f"{folder}/{MODEL}/{path.name}")
    for path in (pilot_path, grid_note, log_path, CONFIGS / f"{MODEL}.yaml"):
        archive.write(path, f"colab/{MODEL}/{path.name}")
print(out, f"{out.stat().st_size / 2**20:.1f} MB", "sha256", hashlib.sha256(out.read_bytes()).hexdigest())
print("DONE. Part", PART, "is finished. Nothing to send: everything is in the shared folder. Tell Brandon.")

## 8. Further parts (greedy runners only)

Everyone stops after one part, except a runner who has a file named `shotlab_greedy.txt` in their own My Drive. That runner keeps going: once a part is finished it claims the next part nobody holds, with the same ticket rule and the same double check as the first claim, and runs the cells of sections 0 to 7 again for that part.

In [ ]:
if GREEDY:
    import gc

    shell = get_ipython()
    last = max(index for index, source in enumerate(In) if source.startswith("import hashlib"))
    first = max(index for index, source in enumerate(In[:last]) if source.startswith("# Same versions"))
    steps = In[first:last + 1]
    while True:
        PART = next_part()
        if PART is None:
            print("No free part left. This runner is finished.")
            break
        MODEL, direction = PART.split("__")
        DIRECTIONS = [direction]
        print("NEXT PART:", PART)
        gc.collect()
        torch.cuda.empty_cache()
        for source in steps:
            shell.run_cell(source).raise_error()